# EDA: MODIS AOD vs PM2.5 — Correlation, Refinement & Physics Correction

**Data sources:**
- **MODIS AOD** — per-station CSVs from `stations_aod_v3` (MCD19A2 HDF extracts)
- **PM2.5** — per-station hourly CSVs from `station_historical_full`
- **ERA5** — per-station CSV with `RH` (%) and `PBLH` (m) for the physics correction
- **Merge strategy** — `merge_asof` nearest timestamp within ±1 h

**Sections:**
1. Load & Merge
2. Data Quality Overview
3. Filter PM2.5 > threshold
4. Helper: OLS + RANSAC
5. Global Scatter (all stations)
6. Per-Station Correlation Summary
7. Correlation Heatmap
8. OLS vs RANSAC R² Comparison
9. Residual Analysis
10. **Data Refinement Pipeline** (F0 → F1)
11. **Physics-Based AOD Correction** (raw vs corrected, F0 & F1)
12. Summary Export

## 0. Imports & Config

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.metrics import r2_score, mean_absolute_error

warnings.filterwarnings('ignore')
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.titlesize'] = 13

# ── CONFIG ──────────────────────────────────────────────────────────────────
AOD_DIR     = '/home/slow_data/Air_Quality/MODIS_MCD19A2/stations_aod_v3'
AQ_DIR      = '/home/slow_data/Air_Quality/historical_full_v2'
ERA5_DIR    = '/home/slow_data/Air_Quality/weather'
STATIONS_DIR    = '/home/work1/projects/Air_Quality/Masterdata/envisoft_27_stations.csv'

PM25_THRESH = 12         # WHO 24-h guideline (μg/m³)
RANSAC_FRAC = 0.80
AOD_COLS    = ['Optical_Depth_047', 'Optical_Depth_055']
GAMMA       = 0.6        # hygroscopic growth exponent
PBLH_MIN    = 50.0       # minimum PBLH clip (m)
# ────────────────────────────────────────────────────────────────────────────

stations = pd.read_csv(STATIONS_DIR)
aod_files = sorted(glob.glob(os.path.join(AOD_DIR, '*.csv')))
aod_files = [f for f in aod_files if any(station_name in f for station_name in stations['stationName'])]
print(f'Found {len(aod_files)} AOD station CSVs')

## 1. Load & Merge Station Data

In [ ]:
def load_and_merge_station(aod_path: str, aq_dir: str) -> pd.DataFrame:
    """
    Load a station's MODIS AOD CSV and merge with its PM2.5 CSV with data cleaning.
    """
    df_aod = pd.read_csv(aod_path, parse_dates=['timestamp'])

    aod_stem = os.path.basename(aod_path).replace('.csv', '')
    aq_stem  = aod_stem.replace(': ', ' ')
    aq_path  = os.path.join(aq_dir, f"{aq_stem}.csv")
    
    if not os.path.isfile(aq_path):
        return pd.DataFrame()

    # Load and initial rename
    df_aq = pd.read_csv(aq_path, parse_dates=['Timestamp']).rename(columns={'Timestamp': 'aq_timestamp'})

    # --- Data Cleaning ---
    # (1) Eliminate sentinel values
    df_aq = df_aq.replace([-9999, -999, 9999], np.nan)

    # (2) Valid-range filtering for PM2.5
    if 'PM2.5' in df_aq.columns:
        df_aq.loc[~df_aq['PM2.5'].between(0, 500), 'PM2.5'] = np.nan

    # (3) Timestamp deduplication
    # Count non-null fields to decide which duplicate to keep
    df_aq['valid_count'] = df_aq.notnull().sum(axis=1)
    df_aq = (df_aq.sort_values(['aq_timestamp', 'valid_count'], ascending=[True, False])
                 .drop_duplicates(subset='aq_timestamp', keep='first')
                 .drop(columns=['valid_count'])
                 .reset_index(drop=True))

    df_aod = df_aod.sort_values('timestamp').reset_index(drop=True)

    # Merge
    df_merged = pd.merge_asof(
        df_aod,
        df_aq, # Including all cleaned columns
        left_on='timestamp',
        right_on='aq_timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1h'),
    )
    df_merged['station_name'] = aod_stem
    return df_merged

frames  = [load_and_merge_station(f, AQ_DIR) for f in aod_files]
df_all  = pd.concat([f for f in frames if not f.empty], ignore_index=True)
df_all['PM2.5'] = pd.to_numeric(df_all['PM2.5'], errors='coerce')
print(f'Merged {df_all["station_name"].nunique()} stations, {len(df_all):,} AOD records')

## 2. Data Quality Overview

In [ ]:
key_cols = ['PM2.5', 'Optical_Depth_047', 'Optical_Depth_055', 'AOD_Uncertainty',
            'Column_WV', 'FineModeFraction', 'AngstromExp_470-780']

missing = df_all[key_cols].isnull().mean().mul(100).round(2).rename('% missing')
print(missing.to_string())

valid_pm25 = df_all['PM2.5'].dropna()
print(f'\nTotal records         : {len(df_all):,}')
print(f'Records with PM2.5    : {valid_pm25.size:,}')
print(f'Date range            : {df_all["timestamp"].min().date()} → {df_all["timestamp"].max().date()}')
print(f'PM2.5 range           : {valid_pm25.min():.1f} – {valid_pm25.max():.1f} μg/m³')

In [ ]:
df_all = df_all.dropna(subset=['PM2.5'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_all['PM2.5'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(PM25_THRESH, color='red', linestyle='--',
                label=f'PM2.5 threshold = {PM25_THRESH} μg/m³')
axes[0].set_title('PM2.5 Distribution (All Stations)')
axes[0].set_xlabel('PM2.5 (μg/m³)')
axes[0].set_ylabel('Count')
axes[0].legend()

counts = df_all.groupby('station_name').size().sort_values(ascending=False)
axes[1].bar(range(len(counts)), counts.values, color='teal')
axes[1].set_xticks(range(len(counts)))
axes[1].set_xticklabels(counts.index, rotation=45, ha='right', fontsize=7)
axes[1].set_title('Records per Station')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3. Filter: PM2.5 > 12 μg/m³

In [ ]:
df_filt = df_all[
    (df_all['PM2.5'] > PM25_THRESH) &
    df_all['Optical_Depth_047'].notna() &
    df_all['Optical_Depth_055'].notna()
].copy()

print(f'Records after PM2.5 > {PM25_THRESH} μg/m³: {len(df_filt):,}  '
      f'({100*len(df_filt)/len(df_all):.1f}% of total)')
print(f'Stations with qualifying data: {df_filt["station_name"].nunique()}')

## 4. Helper: OLS + RANSAC Fitting

In [ ]:
def fit_ols_ransac(x, y, ransac_frac=0.80):
    """Return OLS and RANSAC fits with R², MAE, slope, intercept, and inlier mask."""
    x = np.array(x).reshape(-1, 1)
    y = np.array(y)

    ols = LinearRegression().fit(x, y)
    y_ols   = ols.predict(x)
    r2_ols  = r2_score(y, y_ols)
    mae_ols = mean_absolute_error(y, y_ols)

    min_s  = max(2, int(len(x) * ransac_frac))
    ransac = RANSACRegressor(estimator=LinearRegression(),
                             min_samples=min_s, random_state=42).fit(x, y)
    inlier_mask  = ransac.inlier_mask_
    y_ransac     = ransac.predict(x)
    r2_ransac    = r2_score(y[inlier_mask], y_ransac[inlier_mask])
    mae_ransac   = mean_absolute_error(y[inlier_mask], y_ransac[inlier_mask])

    return dict(
        ols=ols, ransac=ransac, inlier_mask=inlier_mask,
        y_ols=y_ols, y_ransac=y_ransac,
        r2_ols=r2_ols,    mae_ols=mae_ols,
        r2_ransac=r2_ransac, mae_ransac=mae_ransac,
        slope_ols=float(ols.coef_[0]),         intercept_ols=float(ols.intercept_),
        slope_ransac=float(ransac.estimator_.coef_[0]),
        intercept_ransac=float(ransac.estimator_.intercept_),
    )

print('fit_ols_ransac() defined.')

## 5. Global AOD vs PM2.5 Scatter (All Stations Combined)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, aod_col in zip(axes, AOD_COLS):
    sub  = df_filt[['PM2.5', aod_col]].dropna()
    x_v  = sub[aod_col].values
    y_v  = sub['PM2.5'].values
    fit  = fit_ols_ransac(x_v, y_v, RANSAC_FRAC)
    xs   = np.sort(x_v)

    ax.scatter(x_v[~fit['inlier_mask']], y_v[~fit['inlier_mask']],
               color='lightcoral', alpha=0.4, s=15, label='RANSAC outliers')
    ax.scatter(x_v[fit['inlier_mask']],  y_v[fit['inlier_mask']],
               color='steelblue',  alpha=0.5, s=15, label='RANSAC inliers')
    ax.plot(xs, fit['slope_ols']    * xs + fit['intercept_ols'],
            color='orange', lw=2,
            label=f'OLS    R²={fit["r2_ols"]:.3f}  MAE={fit["mae_ols"]:.1f}')
    ax.plot(xs, fit['slope_ransac'] * xs + fit['intercept_ransac'],
            color='green', lw=2, linestyle='--',
            label=f'RANSAC R²={fit["r2_ransac"]:.3f}  MAE={fit["mae_ransac"]:.1f}')

    ax.set_xlabel(aod_col)
    ax.set_ylabel('PM2.5 (μg/m³)')
    ax.set_title(f'All Stations — {aod_col} vs PM2.5 (>{PM25_THRESH} μg/m³)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 6. Per-Station Correlation Summary

In [ ]:
results = []

for station_name, grp in df_filt.groupby('station_name'):
    for aod_col in AOD_COLS:
        sub = grp[['PM2.5', aod_col]].dropna()
        if len(sub) < 5:
            continue
        x = sub[aod_col].values
        y = sub['PM2.5'].values

        pearson_r,  pearson_p  = stats.pearsonr(x, y)
        spearman_r, spearman_p = stats.spearmanr(x, y)

        try:
            fit = fit_ols_ransac(x, y, RANSAC_FRAC)
        except Exception:
            continue

        results.append(dict(
            station_name=station_name, aod_col=aod_col,
            n=len(sub),   n_inliers=fit['inlier_mask'].sum(),
            pearson_r=round(pearson_r, 3),   pearson_p=round(pearson_p, 4),
            spearman_r=round(spearman_r, 3), spearman_p=round(spearman_p, 4),
            ols_slope=round(fit['slope_ols'], 4),
            ols_intercept=round(fit['intercept_ols'], 3),
            ols_r2=round(fit['r2_ols'], 4),  ols_mae=round(fit['mae_ols'], 3),
            ransac_slope=round(fit['slope_ransac'], 4),
            ransac_intercept=round(fit['intercept_ransac'], 3),
            ransac_r2=round(fit['r2_ransac'], 4), ransac_mae=round(fit['mae_ransac'], 3),
        ))

df_results = pd.DataFrame(results)
print(f'Computed correlations for {len(df_results)} station × AOD combinations')
df_results.sort_values('pearson_r', ascending=False)

## 7. Correlation Heatmap (Pearson R per Station)

In [ ]:
pivot = df_results.pivot(index='station_name', columns='aod_col', values='pearson_r')

fig, ax = plt.subplots(figsize=(8, max(4, len(pivot) * 0.45)))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title(f'Pearson R: AOD vs PM2.5 (>{PM25_THRESH} μg/m³) per Station')
ax.set_xlabel('')
ax.set_ylabel('Station')
plt.tight_layout()
plt.show()

## 8. OLS vs RANSAC R² Comparison (per Station)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, aod_col in zip(axes, AOD_COLS):
    sub   = df_results[df_results['aod_col'] == aod_col].sort_values('pearson_r')
    y_pos = range(len(sub))

    ax.barh([y - 0.2 for y in y_pos], sub['ols_r2'],    height=0.35,
            color='orange', label='OLS R²')
    ax.barh([y + 0.2 for y in y_pos], sub['ransac_r2'], height=0.35,
            color='green',  label='RANSAC R²')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sub['station_name'], fontsize=7)
    ax.set_xlabel('R²')
    ax.set_title(f'{aod_col}: OLS vs RANSAC R²')
    ax.axvline(0, color='black', lw=0.8)
    ax.legend()

plt.tight_layout()
plt.show()

## 9. Residual Analysis

In [ ]:
best_aod = df_results.groupby('aod_col')['pearson_r'].median().idxmax()
print(f'Best AOD variable by median Pearson R: {best_aod}')

sub = df_filt[['PM2.5', best_aod]].dropna()
x   = sub[best_aod].values
y   = sub['PM2.5'].values
fit = fit_ols_ransac(x, y, RANSAC_FRAC)

residuals_ols    = y - fit['y_ols']
residuals_ransac = y - fit['y_ransac']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Residual Analysis — {best_aod} vs PM2.5 (All Stations, >{PM25_THRESH} μg/m³)', fontsize=13)

axes[0].scatter(fit['y_ols'],    residuals_ols,
                alpha=0.4, s=12, color='orange', label='OLS')
axes[0].scatter(fit['y_ransac'][fit['inlier_mask']],
                residuals_ransac[fit['inlier_mask']],
                alpha=0.4, s=12, color='green',  label='RANSAC inliers')
axes[0].axhline(0, color='black', lw=1)
axes[0].set_xlabel('Fitted PM2.5 (μg/m³)')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Fitted')
axes[0].legend()

axes[1].hist(residuals_ols,                    bins=40, alpha=0.6, color='orange', label='OLS')
axes[1].hist(residuals_ransac[fit['inlier_mask']], bins=40, alpha=0.6, color='green',  label='RANSAC inliers')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')
axes[1].legend()

stats.probplot(residuals_ols, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot (OLS Residuals)')

plt.tight_layout()
plt.show()

## 10. Data Refinement Pipeline

Progressive filtering to assess the marginal contribution of each quality criterion to the AOD–PM2.5 correlation.

| Stage | Description | Criteria |
|-------|-------------|----------|
| **F0** | No filter | All collocated AOD–PM2.5 pairs |
| **F1** | Fine-mode ratio | FineModeFraction ≥ 0.5 (retains fine-mode dominated aerosol) |
| **F2** | Strict retrieval uncertainty | F1 + AOD_Uncertainty ≤ 0.5 (removes low-confidence retrievals) |

**Evaluation strategy**: metrics are computed *per station*, then averaged across stations (mean ± std). This avoids high-record stations dominating the global fit and gives a more representative summary of AOD–PM2.5 skill.

In [ ]:
# ── Stage definitions ─────────────────────────────────────────────────────────
STAGES = {
    'F0': {'label': 'F0 – No filter',                        'filter': None},
    'F1': {'label': 'F1 – FineModeFraction ≥ 0.5',           'filter': lambda df: df['FineModeFraction'] >= 0.5},
    'F2': {'label': 'F2 – F1 + AOD_Uncertainty ≤ 0.5',       'filter': lambda df: (df['FineModeFraction'] >= 0.5) & (df['AOD_Uncertainty'] <= 0.5)},
}

def apply_stage(df, stage_filter):
    return df.copy() if stage_filter is None else df[stage_filter(df)].copy()

for k, v in STAGES.items():
    df_s = apply_stage(df_filt, v['filter'])
    print(f"{k}: {len(df_s):,} records, {df_s['station_name'].nunique()} stations")

In [ ]:
pipeline_results = []

for stage_key, stage_info in STAGES.items():
    df_stage = apply_stage(df_filt, stage_info['filter'])
    for aod_col in AOD_COLS:
        station_metrics = []
        for stn, grp in df_stage.groupby('station_name'):
            sub = grp[['PM2.5', aod_col]].dropna()
            if len(sub) < 5:
                continue
            x, y = sub[aod_col].values, sub['PM2.5'].values
            pearson_r,  _ = stats.pearsonr(x, y)
            spearman_r, _ = stats.spearmanr(x, y)
            try:
                fit = fit_ols_ransac(x, y, RANSAC_FRAC)
            except Exception:
                continue
            station_metrics.append(dict(
                n=len(sub),
                pearson_r=pearson_r, spearman_r=spearman_r,
                ols_r2=fit['r2_ols'], ols_mae=fit['mae_ols'],
                ransac_r2=fit['r2_ransac'], ransac_mae=fit['mae_ransac'],
            ))

        if not station_metrics:
            continue

        df_stn = pd.DataFrame(station_metrics)
        pipeline_results.append(dict(
            stage=stage_key, label=stage_info['label'], aod_col=aod_col,
            n_stations    =len(df_stn),
            total_n       =int(df_stn['n'].sum()),
            pearson_r     =round(df_stn['pearson_r'].mean(),  3),
            pearson_r_std =round(df_stn['pearson_r'].std(),   3),
            spearman_r    =round(df_stn['spearman_r'].mean(), 3),
            spearman_r_std=round(df_stn['spearman_r'].std(),  3),
            ols_r2        =round(df_stn['ols_r2'].mean(),     4),
            ols_r2_std    =round(df_stn['ols_r2'].std(),      4),
            ols_mae       =round(df_stn['ols_mae'].mean(),    3),
            ransac_r2     =round(df_stn['ransac_r2'].mean(),  4),
            ransac_r2_std =round(df_stn['ransac_r2'].std(),   4),
            ransac_mae    =round(df_stn['ransac_mae'].mean(), 3),
        ))

df_pipeline = pd.DataFrame(pipeline_results)
print(f'Pipeline evaluation — mean across stations:')
df_pipeline

In [ ]:
# ── Scatter plots per stage & AOD column (annotated with mean per-station metrics) ──
n_stages = len(STAGES)
fig, axes = plt.subplots(n_stages, len(AOD_COLS), figsize=(16, 5 * n_stages))
if n_stages == 1:
    axes = [axes]

for row, (stage_key, stage_info) in enumerate(STAGES.items()):
    df_stage = apply_stage(df_filt, stage_info['filter'])
    for col, aod_col in enumerate(AOD_COLS):
        ax  = axes[row][col]
        sub = df_stage[['PM2.5', aod_col]].dropna()
        if len(sub) < 5:
            ax.text(0.5, 0.5, 'Not enough data', ha='center', va='center',
                    transform=ax.transAxes)
            ax.set_title(f'{stage_key} — {aod_col}')
            continue

        ax.scatter(sub[aod_col].values, sub['PM2.5'].values,
                   alpha=0.35, s=10, color='steelblue')

        # Annotate with mean ± std from df_pipeline
        m = df_pipeline[
            (df_pipeline['stage'] == stage_key) & (df_pipeline['aod_col'] == aod_col)
        ]
        if not m.empty:
            r = m.iloc[0]
            info = (
                f"Mean per-station (n={int(r['n_stations'])} stns)\n"
                f"Pearson R  = {r['pearson_r']:.3f} ± {r['pearson_r_std']:.3f}\n"
                f"OLS R²     = {r['ols_r2']:.3f} ± {r['ols_r2_std']:.3f}\n"
                f"RANSAC R²  = {r['ransac_r2']:.3f} ± {r['ransac_r2_std']:.3f}"
            )
            ax.text(0.97, 0.03, info, transform=ax.transAxes, fontsize=8,
                    va='bottom', ha='right',
                    bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.85))

        ax.set_xlabel(aod_col)
        ax.set_ylabel('PM2.5 (μg/m³)')
        ax.set_title(f'{stage_info["label"]} — {aod_col}  (n={len(sub):,})')

plt.tight_layout()
plt.show()

In [ ]:
# ── Summary: mean ± std per-station metric progression across stages ──────────
metrics = [
    ('pearson_r',  'pearson_r_std'),
    ('ols_r2',     'ols_r2_std'),
    ('ransac_r2',  'ransac_r2_std'),
]
fig, axes = plt.subplots(1, len(metrics), figsize=(18, 5))
fig.suptitle('Data Refinement Pipeline — Mean per-station Metric Progression (± 1 std)', fontsize=13)

for ax, (metric, std_col) in zip(axes, metrics):
    for aod_col in AOD_COLS:
        sub = df_pipeline[df_pipeline['aod_col'] == aod_col]
        ax.errorbar(sub['stage'], sub[metric], yerr=sub[std_col],
                    marker='o', capsize=4, label=aod_col)
    ax.set_xlabel('Stage')
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

## 11. Physics-Based AOD Correction

Raw columnar AOD integrates aerosol extinction over the full atmospheric column. To approximate surface-level aerosol concentration we apply:

$$\text{AOD}_{\text{corr}} = \text{AOD} \times \frac{(1 - \text{RH}/100)^{\gamma}}{\text{PBLH}}$$

where  
- $\gamma = 0.6$ — hygroscopic growth exponent  
- PBLH clipped to a minimum of 50 m  
- RH (%) and PBLH (m) from ERA5 reanalysis, matched per station within ±3 h

We evaluate correction benefit on both **F0** (raw) and **F1** (FineModeFraction ≥ 0.5) subsets.

In [ ]:
# ── Load ERA5 RH & PBLH (per-station CSVs), merge with df_filt ──────────────
# Filename convention: weather_{aq_stem}.csv
# Columns used: Timestamp → timestamp, Humidity → RH (%), PBLH (m)

merged_parts = []
missing_era5 = []

for stn, grp in df_filt.groupby('station_name'):
    aq_stem   = stn.replace(': ', ' ')
    era5_path = os.path.join(ERA5_DIR, f'weather_{aq_stem}.csv')
    if not os.path.isfile(era5_path):
        missing_era5.append(stn)
        continue

    # Load ERA5 data
    df_era5_stn = pd.read_csv(era5_path, parse_dates=['Timestamp']).rename(columns={'Timestamp': 'timestamp', 'Humidity': 'RH'})

    # (1) Eliminate sentinel values
    df_era5_stn = df_era5_stn.replace([-9999, -999, 9999], np.nan)

    # (2) Valid-range filtering
    range_limits = {
        'Temperature': (-10, 50),
        'RH': (0, 100),
        'Pressure': (900, 1100),
        'Wind Speed': (0, 50)
    }

    for col, (low, high) in range_limits.items():
        if col in df_era5_stn.columns:
            df_era5_stn.loc[~df_era5_stn[col].between(low, high), col] = np.nan

    # (3) Timestamp deduplication
    # Calculate number of valid meteorological fields
    df_era5_stn['valid_count'] = df_era5_stn.notnull().sum(axis=1)
    df_era5_stn = (df_era5_stn.sort_values(['timestamp', 'valid_count'], ascending=[True, False])
                            .drop_duplicates(subset='timestamp', keep='first')
                            .drop(columns=['valid_count']))

    # Final selection of required columns
    df_era5_stn = df_era5_stn[['timestamp', 'RH', 'PBLH']].sort_values('timestamp')

    m = pd.merge_asof(
        grp.sort_values('timestamp'),
        df_era5_stn,
        on='timestamp', direction='nearest',
        tolerance=pd.Timedelta('3h'),
    )
    merged_parts.append(m)

if missing_era5:
    print(f'ERA5 not found for {len(missing_era5)} station(s): {missing_era5[:3]}...')

if merged_parts:
    df_corr_base = pd.concat(merged_parts, ignore_index=True)
    df_corr_base['PBLH'] = df_corr_base['PBLH'].clip(lower=PBLH_MIN)
    df_corr_base = df_corr_base.dropna(subset=['RH', 'PBLH'])
    print(f'ERA5 matched: {len(df_corr_base):,} records, '
          f'{df_corr_base["station_name"].nunique()} stations')
else:
    print('No ERA5 data matched. Using synthetic RH/PBLH for demonstration.')
    rng = np.random.default_rng(42)
    df_corr_base = df_filt.copy()
    df_corr_base['RH']   = rng.uniform(30, 80,   size=len(df_corr_base))
    df_corr_base['PBLH'] = rng.uniform(200, 1500, size=len(df_corr_base)).clip(min=PBLH_MIN)

In [ ]:
# ── Apply correction: AOD_corr = AOD × (1 − RH/100)^γ / PBLH ─────────────────
AOD_COLS_CORR = []
for col in AOD_COLS:
    corr_col = col + '_corrected'
    df_corr_base[corr_col] = (
        df_corr_base[col]
        * (1 - df_corr_base['RH'] / 100) ** GAMMA
        / df_corr_base['PBLH']
    )
    AOD_COLS_CORR.append(corr_col)

print('Corrected columns created:', AOD_COLS_CORR)
df_corr_base[['station_name', 'PM2.5', 'RH', 'PBLH'] + AOD_COLS + AOD_COLS_CORR].describe().round(4)

In [ ]:
# ── Evaluate correction on F0 and F1 — mean per-station metrics ──────────────
corr_eval_stages = {
    'F0': df_corr_base.copy(),
    'F1': df_corr_base[df_corr_base['FineModeFraction'] >= 0.5].copy(),
}

corr_results = []
for stage_key, df_s in corr_eval_stages.items():
    for aod_col in AOD_COLS:
        for corrected in [False, True]:
            use_col = aod_col + ('_corrected' if corrected else '')
            station_metrics = []
            for stn, grp in df_s.groupby('station_name'):
                sub = grp[['PM2.5', use_col]].dropna()
                if len(sub) < 5:
                    continue
                x, y = sub[use_col].values, sub['PM2.5'].values
                pearson_r,  _ = stats.pearsonr(x, y)
                spearman_r, _ = stats.spearmanr(x, y)
                try:
                    fit = fit_ols_ransac(x, y, RANSAC_FRAC)
                except Exception:
                    continue
                station_metrics.append(dict(
                    n=len(sub),
                    pearson_r=pearson_r, spearman_r=spearman_r,
                    ols_r2=fit['r2_ols'], ols_mae=fit['mae_ols'],
                    ransac_r2=fit['r2_ransac'], ransac_mae=fit['mae_ransac'],
                ))

            if not station_metrics:
                continue

            df_stn = pd.DataFrame(station_metrics)
            corr_results.append(dict(
                stage=stage_key, aod_col=aod_col,
                corrected='Yes' if corrected else 'No',
                n_stations    =len(df_stn),
                total_n       =int(df_stn['n'].sum()),
                pearson_r     =round(df_stn['pearson_r'].mean(),  3),
                pearson_r_std =round(df_stn['pearson_r'].std(),   3),
                spearman_r    =round(df_stn['spearman_r'].mean(), 3),
                spearman_r_std=round(df_stn['spearman_r'].std(),  3),
                ols_r2        =round(df_stn['ols_r2'].mean(),     4),
                ols_r2_std    =round(df_stn['ols_r2'].std(),      4),
                ols_mae       =round(df_stn['ols_mae'].mean(),    3),
                ransac_r2     =round(df_stn['ransac_r2'].mean(),  4),
                ransac_r2_std =round(df_stn['ransac_r2'].std(),   4),
                ransac_mae    =round(df_stn['ransac_mae'].mean(), 3),
            ))

df_corr_results = pd.DataFrame(corr_results)
print('Physics correction evaluation — mean per-station:')
df_corr_results

In [ ]:
# ── Scatter: Raw vs Corrected, F0 vs F1 (annotated with mean per-station metrics) ──
fig, axes = plt.subplots(2, len(AOD_COLS), figsize=(16, 12))
fig.suptitle('Physics-Based AOD Correction: Raw vs Corrected\n(scatter = all points; metrics = mean ± 1 std across stations)', fontsize=13)

for row_i, (stage_key, df_s) in enumerate(corr_eval_stages.items()):
    for col_i, aod_col in enumerate(AOD_COLS):
        ax = axes[row_i][col_i]

        for corrected, pt_c, lbl in [
            (False, 'steelblue',    'Raw'),
            (True,  'mediumpurple', 'Corrected'),
        ]:
            use_col = aod_col + ('_corrected' if corrected else '')
            sub = df_s[['PM2.5', use_col]].dropna()
            if len(sub) < 5:
                continue
            ax.scatter(sub[use_col].values, sub['PM2.5'].values,
                       alpha=0.2, s=10, color=pt_c, label=lbl)

        # Annotation box: mean ± std for raw then corrected
        lines = []
        for corrected, lbl in [(False, 'Raw'), (True, 'Corr')]:
            mask = (
                (df_corr_results['stage']     == stage_key) &
                (df_corr_results['aod_col']   == aod_col)   &
                (df_corr_results['corrected'] == ('Yes' if corrected else 'No'))
            )
            row = df_corr_results[mask]
            if row.empty:
                continue
            m = row.iloc[0]
            lines.append(
                f"[{lbl}] n={int(m['n_stations'])} stns\n"
                f"  Pearson R = {m['pearson_r']:.3f} ± {m['pearson_r_std']:.3f}\n"
                f"  OLS R²    = {m['ols_r2']:.3f} ± {m['ols_r2_std']:.3f}\n"
                f"  RANSAC R² = {m['ransac_r2']:.3f} ± {m['ransac_r2_std']:.3f}"
            )
        if lines:
            ax.text(0.97, 0.03, '\n'.join(lines), transform=ax.transAxes, fontsize=7.5,
                    va='bottom', ha='right',
                    bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.88))

        ax.set_xlabel(aod_col)
        ax.set_ylabel('PM2.5 (μg/m³)')
        ax.set_title(f'{stage_key} — {aod_col}')
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Summary bar: mean ± std per-station metrics, raw vs corrected (F0 & F1) ──
metrics_corr = [
    ('pearson_r',  'pearson_r_std'),
    ('ols_r2',     'ols_r2_std'),
    ('ransac_r2',  'ransac_r2_std'),
]
fig, axes = plt.subplots(1, len(metrics_corr), figsize=(18, 5))
fig.suptitle('Physics-Based Correction — Mean per-station Metric Comparison ± 1 std (F0 & F1)', fontsize=13)

for ax, (metric, std_col) in zip(axes, metrics_corr):
    bar_labels, bar_vals, bar_errs, bar_colors = [], [], [], []
    for stage_key in ['F0', 'F1']:
        for aod_col in AOD_COLS:
            for corr_flag, color in [('No', 'steelblue'), ('Yes', 'tomato')]:
                mask = (
                    (df_corr_results['stage']     == stage_key) &
                    (df_corr_results['aod_col']   == aod_col)   &
                    (df_corr_results['corrected'] == corr_flag)
                )
                row = df_corr_results[mask]
                if row.empty:
                    continue
                bar_labels.append(
                    f'{stage_key}\n{aod_col.split("_")[-1]}\n{"Corr" if corr_flag=="Yes" else "Raw"}'
                )
                bar_vals.append(float(row[metric].iloc[0]))
                bar_errs.append(float(row[std_col].iloc[0]))
                bar_colors.append(color)

    x_pos = range(len(bar_labels))
    ax.bar(x_pos, bar_vals, yerr=bar_errs, color=bar_colors,
           edgecolor='white', capsize=4, error_kw=dict(elinewidth=1.2))
    ax.set_xticks(x_pos)
    ax.set_xticklabels(bar_labels, fontsize=7)
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.axhline(0, color='black', lw=0.6)
    ax.legend(handles=[
        plt.Rectangle((0,0),1,1, color='steelblue', label='Raw'),
        plt.Rectangle((0,0),1,1, color='tomato',    label='Corrected'),
    ], fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Add per-station statistics after F1 and physics correction
per_station_f1_corr = []

for station_name, grp in df_corr_base.groupby('station_name'):
    grp_f1 = grp[grp['FineModeFraction'] >= 0.5]
    
    for aod_col in AOD_COLS:
        for corrected in [False, True]:
            use_col = aod_col + ('_corrected' if corrected else '')
            sub = grp_f1[['PM2.5', use_col]].dropna()
            
            if len(sub) < 5:
                continue
            
            x = sub[use_col].values
            y = sub['PM2.5'].values
            
            pearson_r, pearson_p = stats.pearsonr(x, y)
            spearman_r, spearman_p = stats.spearmanr(x, y)
            fit = fit_ols_ransac(x, y, RANSAC_FRAC)
            
            per_station_f1_corr.append(dict(
                station_name=station_name,
                aod_col=aod_col,
                corrected='Yes' if corrected else 'No',
                n=len(sub),
                n_inliers=fit['inlier_mask'].sum(),
                pearson_r=round(pearson_r, 3),
                pearson_p=round(pearson_p, 4),
                spearman_r=round(spearman_r, 3),
                spearman_p=round(spearman_p, 4),
                ols_r2=round(fit['r2_ols'], 4),
                ols_mae=round(fit['mae_ols'], 3),
                ols_slope=round(fit['slope_ols'], 6),
                ols_intercept=round(fit['intercept_ols'], 4),
                ransac_r2=round(fit['r2_ransac'], 4),
                ransac_mae=round(fit['mae_ransac'], 3),
                ransac_slope=round(fit['slope_ransac'], 6),
                ransac_intercept=round(fit['intercept_ransac'], 4),
            ))

df_results_f1_corr = pd.DataFrame(per_station_f1_corr)
print(f'Per-station F1 + Physics Correction: {len(df_results_f1_corr)} combinations')
df_results_f1_corr.sort_values('pearson_r', ascending=False)

## 12. Summary Export

In [ ]:
Out_dir = "/home/slow_data/Air_Quality/MODIS_MCD19A2/outputs"

In [ ]:
# Baseline per-station stats
out_station = os.path.join(Out_dir, 'correlation_summary_pm25.csv')
df_results_f1_corr.to_csv(out_station, index=False)
print(f'Per-station summary   → {out_station}')

# Pipeline summary (F0, F1)
out_pipeline = os.path.join(Out_dir, 'pipeline_summary.csv')
df_pipeline.to_csv(out_pipeline, index=False)
print(f'Pipeline summary      → {out_pipeline}')

# Physics correction summary
out_corr = os.path.join(Out_dir, 'physics_correction_summary.csv')
df_corr_results.to_csv(out_corr, index=False)
print(f'Physics corr summary  → {out_corr}')